In [1]:
from typing import Dict, List, Any, Optional, Union, Annotated
from pydantic import BaseModel, Field, validator, field_validator
from enum import Enum
from datetime import datetime
import asyncio
import uuid

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor
from langgraph.checkpoint.memory import MemorySaver

# Pydantic Models
class TaskType(str, Enum):
    SEARCH_SEMANTIC = "search_semantic"
    SEARCH_SYNTACTIC = "search_syntactic"
    REASONING = "reasoning"
    SYNTHESIS = "synthesis"
    VALIDATION = "validation"
    PLANNING = "planning"

class TaskStatus(str, Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"

class Task(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    type: TaskType
    content: str
    priority: int = Field(default=1, ge=1, le=10)
    context: Dict[str, Any] = Field(default_factory=dict)
    status: TaskStatus = Field(default=TaskStatus.PENDING)
    created_at: datetime = Field(default_factory=datetime.now)
    parent_task_id: Optional[str] = None
    
    class Config:
        use_enum_values = True

class AgentResponse(BaseModel):
    agent_id: str
    task_id: str
    result: Any
    confidence: float = Field(ge=0.0, le=1.0)
    metadata: Dict[str, Any] = Field(default_factory=dict)
    processing_time: float = Field(default=0.0)
    created_at: datetime = Field(default_factory=datetime.now)
    
    @field_validator('confidence')
    def validate_confidence(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError('Confidence must be between 0 and 1')
        return v

class AgentCapability(BaseModel):
    task_type: TaskType
    proficiency: float = Field(ge=0.0, le=1.0)
    max_concurrent_tasks: int = Field(default=1, ge=1)

class AgentProfile(BaseModel):
    agent_id: str
    name: str
    description: str
    capabilities: List[AgentCapability]
    is_active: bool = Field(default=True)
    current_load: int = Field(default=0)
    
    def can_handle(self, task: Task) -> bool:
        return any(cap.task_type == task.type for cap in self.capabilities)
    
    def get_proficiency(self, task_type: TaskType) -> float:
        for cap in self.capabilities:
            if cap.task_type == task_type:
                return cap.proficiency
        return 0.0
    
    def is_available(self) -> bool:
        max_load = max([cap.max_concurrent_tasks for cap in self.capabilities], default=1)
        return self.is_active and self.current_load < max_load

# LangGraph State
class NetworkState(BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    user_input: str = ""
    current_task: Optional[Task] = None
    task_queue: List[Task] = Field(default_factory=list)
    completed_tasks: List[Task] = Field(default_factory=list)
    agent_responses: List[AgentResponse] = Field(default_factory=list)
    context: Dict[str, Any] = Field(default_factory=dict)
    final_response: str = ""
    iteration_count: int = 0
    max_iterations: int = 10
    
    class Config:
        arbitrary_types_allowed = True

# Agent Implementations
class BaseAgentNode:
    def __init__(self, profile: AgentProfile):
        self.profile = profile
        
    async def execute(self, state: NetworkState) -> NetworkState:
        """Execute agent logic"""
        if not state.current_task or not self.profile.can_handle(state.current_task):
            return state
            
        start_time = datetime.now()
        
        try:
            # Process the task
            result = await self._process_task(state.current_task, state.context)
            processing_time = (datetime.now() - start_time).total_seconds()
            
            # Create response
            response = AgentResponse(
                agent_id=self.profile.agent_id,
                task_id=state.current_task.id,
                result=result,
                confidence=self._calculate_confidence(state.current_task, result),
                processing_time=processing_time,
                metadata=self._get_metadata(state.current_task, result)
            )
            
            # Update state
            state.agent_responses.append(response)
            state.current_task.status = TaskStatus.COMPLETED
            state.completed_tasks.append(state.current_task)
            
            # Add result to context for other agents
            context_key = f"{self.profile.agent_id}_result"
            state.context[context_key] = result
            
        except Exception as e:
            state.current_task.status = TaskStatus.FAILED
            state.context["error"] = str(e)
            
        return state
    
    async def _process_task(self, task: Task, context: Dict[str, Any]) -> Any:
        """Override this method in specific agent implementations"""
        raise NotImplementedError
    
    def _calculate_confidence(self, task: Task, result: Any) -> float:
        """Calculate confidence based on task and result"""
        base_proficiency = self.profile.get_proficiency(TaskType(task.type))
        # Add logic based on result quality, context, etc.
        return min(base_proficiency + 0.1, 1.0)
    
    def _get_metadata(self, task: Task, result: Any) -> Dict[str, Any]:
        """Get metadata for the response"""
        return {
            "agent_name": self.profile.name,
            "task_type": task.type,
            "result_type": type(result).__name__
        }

class SemanticSearchAgent(BaseAgentNode):
    async def _process_task(self, task: Task, context: Dict[str, Any]) -> Dict[str, Any]:
        # Simulate semantic search processing
        await asyncio.sleep(0.1)
        
        # Mock semantic search with vector similarity
        search_results = {
            "query": task.content,
            "documents": [
                {
                    "id": f"doc_{i}",
                    "content": f"Semantic content related to: {task.content[:50]}...",
                    "similarity_score": 0.9 - (i * 0.1),
                    "source": f"knowledge_base_{i}"
                }
                for i in range(3)
            ],
            "search_method": "vector_similarity",
            "total_results": 3
        }
        
        return search_results

class SyntacticSearchAgent(BaseAgentNode):
    async def _process_task(self, task: Task, context: Dict[str, Any]) -> Dict[str, Any]:
        await asyncio.sleep(0.08)
        
        # Mock syntactic search with keyword matching
        search_results = {
            "query": task.content,
            "documents": [
                {
                    "id": f"syntax_doc_{i}",
                    "content": f"Syntactic match for: {task.content[:50]}...",
                    "match_score": 0.85 - (i * 0.15),
                    "matched_terms": task.content.split()[:3]
                }
                for i in range(2)
            ],
            "search_method": "keyword_matching",
            "total_results": 2
        }
        
        return search_results

class ReasoningAgent(BaseAgentNode):
    async def _process_task(self, task: Task, context: Dict[str, Any]) -> Dict[str, Any]:
        await asyncio.sleep(0.15)
        
        # Use search results from context if available
        semantic_results = context.get("semantic_search_result", {})
        syntactic_results = context.get("syntactic_search_result", {})
        
        reasoning_chain = [
            "Analyzed semantic search results",
            "Integrated syntactic patterns",
            "Applied logical inference rules",
            "Evaluated evidence quality",
            "Formed coherent conclusions"
        ]
        
        return {
            "reasoning_chain": reasoning_chain,
            "conclusion": f"Based on analysis of '{task.content}', the reasoning suggests...",
            "evidence_sources": len(semantic_results.get("documents", [])) + len(syntactic_results.get("documents", [])),
            "logical_strength": 0.87
        }

class SynthesisAgent(BaseAgentNode):
    async def _process_task(self, task: Task, context: Dict[str, Any]) -> Dict[str, Any]:
        await asyncio.sleep(0.12)
        
        # Gather all previous results
        semantic_result = context.get("semantic_search_result", {})
        syntactic_result = context.get("syntactic_search_result", {})
        reasoning_result = context.get("reasoning_result", {})
        
        # Count total sources
        total_sources = (
            len(semantic_result.get("documents", [])) +
            len(syntactic_result.get("documents", [])) +
            bool(reasoning_result)
        )
        
        synthesis = {
            "unified_response": self._create_unified_response(task.content, context),
            "sources_integrated": total_sources,
            "coherence_score": 0.92,
            "synthesis_method": "multi_agent_fusion",
            "component_results": {
                "semantic_search": bool(semantic_result),
                "syntactic_search": bool(syntactic_result),
                "reasoning": bool(reasoning_result)
            }
        }
        
        return synthesis
    
    def _create_unified_response(self, query: str, context: Dict[str, Any]) -> str:
        components = []
        
        if "semantic_search_result" in context:
            components.append("semantic analysis")
        if "syntactic_search_result" in context:
            components.append("syntactic matching")
        if "reasoning_result" in context:
            components.append("logical reasoning")
            
        return f"Comprehensive response to '{query}' synthesized from {', '.join(components)}"

# LangGraph Network Implementation
class AgenticNetwork:
    def __init__(self):
        self.agents = {}
        self.memory = MemorySaver()
        self.graph = None
        self._setup_agents()
        self._build_graph()
    
    def _setup_agents(self):
        """Initialize specialized agents"""
        
        # Semantic Search Agent
        semantic_profile = AgentProfile(
            agent_id="semantic_search",
            name="Semantic Search Agent",
            description="Performs vector-based semantic search",
            capabilities=[
                AgentCapability(task_type=TaskType.SEARCH_SEMANTIC, proficiency=0.9, max_concurrent_tasks=3)
            ]
        )
        self.agents["semantic_search"] = SemanticSearchAgent(semantic_profile)
        
        # Syntactic Search Agent
        syntactic_profile = AgentProfile(
            agent_id="syntactic_search",
            name="Syntactic Search Agent", 
            description="Performs keyword-based syntactic search",
            capabilities=[
                AgentCapability(task_type=TaskType.SEARCH_SYNTACTIC, proficiency=0.85, max_concurrent_tasks=3)
            ]
        )
        self.agents["syntactic_search"] = SyntacticSearchAgent(syntactic_profile)
        
        # Reasoning Agent
        reasoning_profile = AgentProfile(
            agent_id="reasoning",
            name="Reasoning Agent",
            description="Performs logical reasoning and inference",
            capabilities=[
                AgentCapability(task_type=TaskType.REASONING, proficiency=0.88, max_concurrent_tasks=2)
            ]
        )
        self.agents["reasoning"] = ReasoningAgent(reasoning_profile)
        
        # Synthesis Agent
        synthesis_profile = AgentProfile(
            agent_id="synthesis",
            name="Synthesis Agent",
            description="Synthesizes results from multiple agents",
            capabilities=[
                AgentCapability(task_type=TaskType.SYNTHESIS, proficiency=0.92, max_concurrent_tasks=1)
            ]
        )
        self.agents["synthesis"] = SynthesisAgent(synthesis_profile)
    
    def _build_graph(self):
        """Build the LangGraph workflow"""
        
        # Define the graph
        workflow = StateGraph(NetworkState)
        
        # Add nodes
        workflow.add_node("planner", self._planner_node)
        workflow.add_node("parallel_search", self._parallel_search_node)
        workflow.add_node("reasoning", self._reasoning_node)
        workflow.add_node("synthesis", self._synthesis_node)
        workflow.add_node("validator", self._validator_node)
        
        # Define edges
        workflow.set_entry_point("planner")
        workflow.add_edge("planner", "parallel_search")
        workflow.add_edge("parallel_search", "reasoning")
        workflow.add_edge("reasoning", "synthesis")
        workflow.add_edge("synthesis", "validator")
        workflow.add_conditional_edges(
            "validator",
            self._should_continue,
            {
                "continue": "reasoning",  # Iterate if needed
                "end": END
            }
        )
        
        # Compile the graph
        self.graph = workflow.compile(checkpointer=self.memory)
    
    async def _planner_node(self, state: NetworkState) -> NetworkState:
        """Planning phase - break down user input into tasks"""
        
        # Create execution plan based on user input
        tasks = [
            Task(
                type=TaskType.SEARCH_SEMANTIC,
                content=state.user_input,
                priority=1
            ),
            Task(
                type=TaskType.SEARCH_SYNTACTIC,
                content=state.user_input,
                priority=1
            ),
            Task(
                type=TaskType.REASONING,
                content=state.user_input,
                priority=2
            ),
            Task(
                type=TaskType.SYNTHESIS,
                content=state.user_input,
                priority=3
            )
        ]
        
        state.task_queue = tasks
        state.messages.append(SystemMessage(content=f"Created execution plan with {len(tasks)} tasks"))
        
        return state
    
    async def _parallel_search_node(self, state: NetworkState) -> NetworkState:
        """Execute search tasks in parallel"""
        
        search_tasks = [task for task in state.task_queue if "search" in task.type]
        
        # Execute search tasks concurrently
        search_coroutines = []
        for task in search_tasks:
            state.current_task = task
            if task.type == TaskType.SEARCH_SEMANTIC:
                search_coroutines.append(self.agents["semantic_search"].execute(state))
            elif task.type == TaskType.SEARCH_SYNTACTIC:
                search_coroutines.append(self.agents["syntactic_search"].execute(state))
        
        if search_coroutines:
            # Wait for all search tasks to complete
            await asyncio.gather(*search_coroutines)
        
        # Remove completed search tasks
        state.task_queue = [task for task in state.task_queue if "search" not in task.type]
        
        return state
    
    async def _reasoning_node(self, state: NetworkState) -> NetworkState:
        """Execute reasoning task"""
        
        reasoning_tasks = [task for task in state.task_queue if task.type == TaskType.REASONING]
        
        if reasoning_tasks:
            state.current_task = reasoning_tasks[0]
            await self.agents["reasoning"].execute(state)
            state.task_queue.remove(reasoning_tasks[0])
        
        return state
    
    async def _synthesis_node(self, state: NetworkState) -> NetworkState:
        """Execute synthesis task"""
        
        synthesis_tasks = [task for task in state.task_queue if task.type == TaskType.SYNTHESIS]
        
        if synthesis_tasks:
            state.current_task = synthesis_tasks[0]
            await self.agents["synthesis"].execute(state)
            state.task_queue.remove(synthesis_tasks[0])
        
        return state
    
    async def _validator_node(self, state: NetworkState) -> NetworkState:
        """Validate results and determine if iteration is needed"""
        
        # Simple validation logic
        synthesis_responses = [r for r in state.agent_responses if r.agent_id == "synthesis"]
        
        if synthesis_responses:
            synthesis_result = synthesis_responses[-1]
            if synthesis_result.confidence > 0.8:
                state.final_response = synthesis_result.result["unified_response"]
            else:
                # Need more iterations
                state.iteration_count += 1
        
        return state
    
    def _should_continue(self, state: NetworkState) -> str:
        """Determine if we should continue iterating or end"""
        
        if (state.final_response and 
            state.iteration_count < state.max_iterations and 
            len(state.task_queue) == 0):
            return "end"
        elif state.iteration_count >= state.max_iterations:
            return "end"
        else:
            return "continue"
    
    async def process_query(self, user_input: str) -> str:
        """Main entry point for processing user queries"""
        
        # Initialize state
        initial_state = NetworkState(
            user_input=user_input,
            messages=[HumanMessage(content=user_input)]
        )
        
        # Create a unique thread for this conversation
        thread_id = str(uuid.uuid4())
        config = {"configurable": {"thread_id": thread_id}}
        
        # Execute the graph
        final_state = None
        async for state in self.graph.astream(initial_state, config=config):
            final_state = list(state.values())[0]
        
        return final_state.final_response or "Unable to generate response"
    
    def get_network_stats(self) -> Dict[str, Any]:
        """Get network performance statistics"""
        return {
            "agents_count": len(self.agents),
            "agent_profiles": {
                agent_id: {
                    "name": agent.profile.name,
                    "capabilities": [cap.task_type for cap in agent.profile.capabilities],
                    "is_active": agent.profile.is_active
                }
                for agent_id, agent in self.agents.items()
            }
        }



/home/octoopt/workspace/projects/learn-from-basics/the-notes/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'ToolExecutor' from 'langgraph.prebuilt' (/home/octoopt/workspace/projects/learn-from-basics/the-notes/.venv/lib/python3.12/site-packages/langgraph/prebuilt/__init__.py)

In [ ]:
# Usage Example
async def demonstrate_langgraph_network():
    """Demonstrate the LangGraph-based agentic network"""
    
    print("🚀 Initializing LangGraph Agentic Network...")
    network = AgenticNetwork()
    
    print("✅ Network initialized with agents:")
    stats = network.get_network_stats()
    for agent_id, info in stats["agent_profiles"].items():
        print(f"  - {info['name']} ({agent_id}): {info['capabilities']}")
    
    print("\n" + "="*60 + "\n")
    
    # Test queries
    test_queries = [
        "What are the environmental impacts of renewable energy?",
        "How does machine learning improve healthcare outcomes?",
        "Explain the economics of sustainable agriculture"
    ]
    
    for query in test_queries:
        print(f"🤔 Processing: {query}")
        print("⚡ Network executing...")
        
        start_time = datetime.now()
        response = await network.process_query(query)
        processing_time = (datetime.now() - start_time).total_seconds()
        
        print(f"✅ Response ({processing_time:.2f}s):")
        print(f"   {response}")
        print("\n" + "-"*60 + "\n")

# Run the demonstration
if __name__ == "__main__":
    asyncio.run(demonstrate_langgraph_network())